# Global Topic Renaming — iGEM Teams (Part 2)

The per-cluster naming in Part 1 works locally: each topic is named in
isolation. This can produce **duplicate or ambiguous names** when two clusters
cover related sub-themes (e.g. both named "History of Synthetic Biology").

This notebook fixes that by giving the LLM a **global view** of all
**iGEM Teams** topics at once. We use **OpenAI function calling** so the
model returns a structured array of `(topic_id, name)` pairs — one per cluster —
guaranteeing distinct, publication-ready names.

Overwrites `teams_topic_names.txt`, adding a `global_name` column.

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 03-topic_names/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from aux.paths import MODELS_DIR, OPENAI_MODEL
from aux.openai_client import load_prompts, make_client
from aux.tables import load_topic_names, save_topic_names
from aux.global_rename import rename_topics_global

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"
MODEL  = OPENAI_MODEL

prompts = load_prompts()
client = make_client()

## 1. Load Part 1 results

In [ ]:
names = load_topic_names(PREFIX)
print(f"Teams: {len(names)} topics")
names[["topic", "name", "description"]].head()

## 2. Global rename (function calling)

In [ ]:
renamed = rename_topics_global(names, client, prompts, model=MODEL)
renamed[["topic", "name", "global_name", "description"]]

## 3. Save final results

In [ ]:
save_topic_names(renamed, PREFIX)
print(f"Saved → {MODELS_DIR / f'{PREFIX}_topic_names.txt'} (added global_name)")